# Session 23: How much faster? Speedup, Amdahl and scaling

In Session 22 a pool of processes made our jobs faster. Today we ask how much faster, and why adding workers stops
helping at some point. We measure one small program with 1, 2, 4, 8, 16 and 32 workers and compare what we measure
with Amdahl's law.

- **Part 1**: strong scaling of tasks that compute.
- **Part 2**: the same program with tasks that read a large array from memory.
- **Part 3**: weak scaling.
- **Part 4**: homework.

Run the cells in order, from top to bottom: click a cell and press Shift+Enter, on Mac and on Windows. When the text
says **Predict**, decide on your answer before you run the next cell.

## Before you start

Look at the top right corner of this notebook. It must say **OSPC (Python 3.12)**. If it says something else, click
on that name and choose OSPC (Python 3.12) from the list.

Every measurement runs as a batch job with 64 CPUs, the most a student account can use (32 cores). Your JupyterLab
session has only 2 CPUs, that is one core, so it is the wrong place to measure speedup. The cell below puts the course
Python in the `PATH`, so that the jobs use it, and loads the libraries for the tables and the charts.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

os.environ["PATH"] = "/data/software/envs/ospc/bin:" + os.environ["PATH"]
print("CPUs of your session:", len(os.sched_getaffinity(0)))

## Part 1. Strong scaling

### The program

The next cell writes `scaling.py`. It does the job of the stats step of Lab 1: many resamples of the order values,
then a confidence interval.

1. **Prepare** (one worker): build and sort 40 million order values. This is the part of the program that only one
   worker can do, the **serial part**.
2. **Pool** (n workers): run the tasks in a `ProcessPoolExecutor` with n workers. There are two kinds of task:
   `compute` works on small arrays that fit in the core's own cache, `data` reads a million orders at random from
   the 320 MB array in memory.
3. **Combine** (one worker): the percentiles of the results.

For each number of workers the script prints one line: the time of each step and the total, in seconds.

In [ ]:
%%writefile scaling.py
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np

ORDERS = int(os.environ.get("ORDERS", 40_000_000))    # orders prepared by one worker
REPEATS = int(os.environ.get("REPEATS", 1))           # runs of every configuration
TASKS = {"compute": 640, "data": 1280}                # the fixed problem (strong scaling)
PER_WORKER = {"compute": 20, "data": 40}              # tasks per worker (weak scaling)

values = None


def prepare():
    # The serial part: one worker builds and sorts the column of order values.
    global values
    rng = np.random.default_rng(2025)
    values = np.sort(rng.gamma(2.0, 30.0, ORDERS))


def compute(seed):
    # All the work happens on small arrays, inside the core's cache.
    rng = np.random.default_rng(seed)
    return np.mean([rng.gamma(2.0, 30.0, 50_000).mean() for _ in range(20)])


def data(seed):
    # One resample: the mean of a million orders picked at random from memory.
    rng = np.random.default_rng(seed)
    return values[rng.integers(0, ORDERS, 1_000_000)].mean()


def run(task, workers, tasks):
    t0 = time.perf_counter()
    prepare()
    t1 = time.perf_counter()
    with ProcessPoolExecutor(workers) as ex:
        means = list(ex.map(task, range(tasks), chunksize=max(1, tasks // (4 * workers))))
    t2 = time.perf_counter()
    low, high = np.percentile(means, [2.5, 97.5])
    t3 = time.perf_counter()
    return t1 - t0, t2 - t1, t3 - t2, t3 - t0


if __name__ == "__main__":
    kind, mode, sizes = sys.argv[1], sys.argv[2], [int(x) for x in sys.argv[3:]]
    task = {"compute": compute, "data": data}[kind]
    print(f"# node {os.uname().nodename}, {len(os.sched_getaffinity(0))} CPUs, job {os.environ.get('SLURM_JOB_ID')}")
    print("workers,tasks,prepare,pool,combine,total", flush=True)
    for _ in range(REPEATS):
        for n in sizes:
            tasks = TASKS[kind] if mode == "strong" else PER_WORKER[kind] * n
            p, c, e, total = run(task, n, tasks)
            print(f"{n},{tasks},{p:.3f},{c:.3f},{e:.3f},{total:.3f}", flush=True)

### Run it on the cluster

**Strong scaling** keeps the problem fixed (640 compute tasks) and adds workers. One batch job runs the program with
1, 2, 4, 8, 16 and 32 workers, one after the other, so every line uses the same node and the same CPUs. `--wait` keeps
the cell busy until the job ends, about 30 seconds.

**Predict**: with 32 workers, how many times faster than with one worker? Write down a number.

In [ ]:
!sbatch -p cpu -c 64 --mem=8G -t 5 -o strong-compute.out --wait --wrap "python scaling.py compute strong 1 2 4 8 16 32"
!cat strong-compute.out

### Speedup and efficiency

The **speedup** with n workers is the time with one worker divided by the time with n workers. The **efficiency** is the
speedup divided by n: 1.0 means that every worker was busy all the time.

The function below reads a file like `strong-compute.out` into a table. With several repeats, it takes the median of
each configuration.

In [ ]:
def results(path):
    # One row per number of workers: median times, speedup and efficiency.
    runs = pd.read_csv(path, comment="#")
    table = runs.groupby("workers")[["prepare", "pool", "total"]].median()
    table["speedup"] = table["total"].iloc[0] / table["total"]
    table["efficiency"] = table["speedup"] / table.index
    return table.round(2)


compute = results("strong-compute.out")
compute

### Amdahl's law

If a fraction `s` of the time with one worker can only run on one worker, n workers can at best give a speedup of

    S(n) = 1 / (s + (1 - s) / n)

and never more than 1 / s, however many workers we add. Here `s` is the prepare time divided by the total time with
one worker. The chart shows the ideal speedup (n workers, n times faster), Amdahl's prediction and what we measured.

**Predict**: will the measured points sit on the ideal line, on Amdahl's curve, or below both?

In [ ]:
def amdahl(s, n):
    return 1 / (s + (1 - s) / n)


def chart(table, label, color):
    s = table["prepare"].iloc[0] / table["total"].iloc[0]
    n = table.index
    plt.plot(n, n, "--", color="#898781", label="ideal")
    plt.plot(n, amdahl(s, n), color="#c3c2b7", linewidth=3, label=f"Amdahl, s = {s:.1%}")
    plt.plot(n, table["speedup"], "o-", color=color, label=f"measured: {label}")
    plt.xscale("log", base=2)
    plt.xticks(n, n)
    plt.xlabel("workers")
    plt.ylabel("speedup")
    plt.legend()
    print(f"serial part s = {s:.1%}: the speedup can never exceed {1 / s:.1f}")


chart(compute, "compute", "#2a78d6")

The measured speedup follows Amdahl's curve and stays a little below it. The small gap at 16 and 32 workers is
**overhead**: starting 32 processes, handing out the tasks, and waiting for the last one to finish.

### The serial fraction from the measurements

We can also go the other way: from a measured speedup S with n workers, estimate the serial fraction

    e = (1/S - 1/n) / (1 - 1/n)

If the only thing that stops the speedup is the serial part, e stays the same for every n, and equals the s we got
by timing the prepare step. Your project asks you to compare these two estimates.

In [ ]:
def serial_fraction(table):
    n = table.index[1:]
    S = table["speedup"].iloc[1:]
    return ((1 / S - 1 / n) / (1 - 1 / n)).round(3)


serial_fraction(compute)

## Part 2. When the workers share the memory

Same program, but now every task reads a million orders at random from the 320 MB array: the work is mostly moving
data from memory to the core. The serial part is about the same as before.

**Predict**: will these tasks follow Amdahl's curve as well as the compute tasks did? The job takes about 40 seconds.

In [ ]:
!sbatch -p cpu -c 64 --mem=8G -t 5 -o strong-data.out --wait --wrap "python scaling.py data strong 1 2 4 8 16 32"
data = results("strong-data.out")
chart(data, "data", "#eb6834")
data

Up to 4 workers the data tasks follow Amdahl. With more workers they fall below the curve: the pool time shrinks much
less than the number of workers grows. All the workers read from the same memory, and the memory cannot deliver data
faster (Lecture 13, "Compute or Move?"). Amdahl's law assumes that the parallel part divides perfectly among the
workers; here it does not. The serial fraction estimated from the measurements grows with n, which is the sign that
something other than the serial part is in the way.

How much the data tasks suffer depends on the node: the first line of the output says where your job ran. On haskell,
on 7 November, the pool time stopped at about 2 seconds from 8 workers on, and the speedup at 4.6; on fortran it
reached 7.3. Always compare measurements made on the same node.

In [ ]:
serial_fraction(data)

## Part 3. Weak scaling

**Weak scaling** grows the problem with the workers: 20 compute tasks (or 40 data tasks) per worker. If the program
scales well, every worker has the same amount of work at every size, and the time stays the same. Here we look at the
pool time only, because the prepare step does not grow.

**Predict**: which kind of task keeps a flat pool time up to 32 workers? Both jobs together take about 20 seconds.

In [ ]:
!sbatch -p cpu -c 64 --mem=8G -t 5 -o weak-compute.out --wait --wrap "python scaling.py compute weak 1 2 4 8 16 32"
!sbatch -p cpu -c 64 --mem=8G -t 5 -o weak-data.out --wait --wrap "python scaling.py data weak 1 2 4 8 16 32"
weak = pd.DataFrame({"compute": results("weak-compute.out")["pool"], "data": results("weak-data.out")["pool"]})
weak.plot(marker="o", color=["#2a78d6", "#eb6834"], logx=True, xticks=weak.index, ylabel="pool time (s)", ylim=(0, None))
weak

## Part 4. Homework

1. Double the serial part: run the compute tasks with 80 million orders. The variable `ORDERS` before `python`
   changes the size of the prepare step. **Predict** first: what is the new serial fraction, and what is the new
   largest possible speedup? Then run the cell and compare.
2. Run Part 1 again with `REPEATS=5` (about two and a half minutes). Look at the spread between the five runs of
   each configuration. Your project asks for the median and the spread of at least five runs.

In [ ]:
!sbatch -p cpu -c 64 --mem=8G -t 5 -o strong-compute-80M.out --wait --wrap "ORDERS=80000000 python scaling.py compute strong 1 2 4 8 16 32"
bigger = results("strong-compute-80M.out")
chart(bigger, "compute, 80 million orders", "#1baf7a")
bigger